# **Transformers Fine-Tune With Hugging Face**

#### **Import Libraries**

In [1]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification


C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Define and Import Dataset** 

In [2]:
dataset = load_dataset("yelp_review_full")
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [3]:
train_dataset = dataset['train']
test_dataset = dataset['test']


In [4]:
train_dataset[0]

{'label': 4,
 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."}

Decrease the amount of data inorder to reduce the training time. 

**warining**: train with less data, the model accuracy will be lower

In [5]:
new_train_dataset = train_dataset.select([i for i in range(1000)])
new_test_dataset = test_dataset.select([i for i in range(500)])

In [6]:
for i in range (20):
    print(new_train_dataset[i])

{'label': 4, 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."}
{'label': 1, 'text': "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  Y

### **Define Tokenization**

In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [8]:
sample_token = tokenizer(new_train_dataset[0]["text"])
sample_token

{'input_ids': [101, 173, 1197, 119, 2284, 2953, 3272, 1917, 178, 1440, 1111, 1107, 170, 1704, 22351, 119, 1119, 112, 188, 3505, 1105, 3123, 1106, 2037, 1106, 1443, 1217, 10063, 4404, 132, 1119, 112, 188, 1579, 1113, 1159, 1107, 3195, 1117, 4420, 132, 1119, 112, 188, 6559, 1114, 170, 1499, 118, 23555, 2704, 113, 183, 9379, 114, 1134, 1139, 2153, 1138, 3716, 1106, 1143, 1110, 1304, 1696, 1107, 1692, 1380, 5940, 1105, 1128, 1444, 6059, 132, 1105, 1128, 1169, 1243, 5991, 16179, 1106, 1267, 18137, 1443, 1515, 1106, 1267, 1140, 1148, 119, 1541, 117, 1184, 1167, 1202, 1128, 1444, 136, 178, 112, 182, 2807, 1303, 1774, 1106, 1341, 1104, 1251, 11344, 178, 1138, 1164, 1140, 117, 1133, 178, 112, 182, 1541, 4619, 170, 9153, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

##### Function to tokenize the dataset texts  

In [9]:
def text_tokenization (dataset):
    return tokenizer(dataset["text"], padding='max_length', truncation=True)

In [10]:
tokenized_train_dataset = new_train_dataset.map(text_tokenization,batched=True)
tokenized_test_dataset = new_test_dataset.map(text_tokenization, batched=True)

In [11]:
tokenized_train_dataset[0]

{'label': 4,
 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank.",
 'input_ids': [101,
  173,
  1197,
  119,
  2284,
  2953,
  3272,
  1917,
  178,
  1440,
  1111,
  1107,
  170,
  1704,
  22351,
  119,
  1119,
  112,
  188,
  3505,
  1105,
  3123,
  1106,
  2037,
  1106,
  1443,
  1217,
  10063,
  4404,
  132,
  1119,
  112,
  188,
  1579,
  1113,
  1159,
  1107,
  3195,
  1117,
  4420,
  132,
  1119,
  112,
  188,
  6559,
  1114,
  170,
  1499,
  118,
  23555,
  2704,
  113,
  183,
  9379,
  114,
  1

In [12]:
for i in range (10):
    print(tokenized_train_dataset[i])

{'label': 4, 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank.", 'input_ids': [101, 173, 1197, 119, 2284, 2953, 3272, 1917, 178, 1440, 1111, 1107, 170, 1704, 22351, 119, 1119, 112, 188, 3505, 1105, 3123, 1106, 2037, 1106, 1443, 1217, 10063, 4404, 132, 1119, 112, 188, 1579, 1113, 1159, 1107, 3195, 1117, 4420, 132, 1119, 112, 188, 6559, 1114, 170, 1499, 118, 23555, 2704, 113, 183, 9379, 114, 1134, 1139, 2153, 1138, 3716, 1106, 1143, 1110, 1304, 1696, 1107, 1692, 1380, 5940, 1105, 1128, 1444, 6059, 132, 

##### **Format the Datasets**

In [13]:
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["text"])

tokenized_train_dataset = tokenized_train_dataset.rename_column("label", "labels")
tokenized_test_dataset = tokenized_test_dataset.rename_column("label", "labels")

#convert lists into tensors
tokenized_train_dataset.set_format("torch")
tokenized_test_dataset.set_format("torch")

In [14]:
tokenized_train_dataset[2]

{'labels': tensor(3),
 'input_ids': tensor([  101, 18511,  1280,  1106,  1987,   119, 20029,  1111,  1166,  1275,
          1201,   119,   146,  1341,   146,  1108,  1141,  1104,  1117,  2198,
          4420,  1165,  1119,  1408,  1120,   150,  3048, 14666,   119,  1124,
           112,   188,  1151,  1632,  1166,  1103,  1201,  1105,  1110,  1541,
          1155,  1164,  1103,  1992,  3439,   119,  1135,  1110,  1272,  1104,
          1140,   117,  1136,  1139,  1208,  1393,   176,  5730,  1987,   119,
          2392,  5792,   117,  1115,   146,  1276,  1149,   146,  1138, 20497,
         12725,  7540,   119,  1124, 16001,  1155,  6665,  1114,  1128,  1105,
          1110,  1304,  5351,  1105,  4287,   119,  1124,  2144,   112,   189,
          3942,  1105,  4390,  1155,  1103,  1268,  3243,   119,  6424, 17213,
          1105,  3349,  1106,  1129,  2023,  1107,  1103,  7812,  1113,  1451,
          7631,  1104,  1240,  2657,  2332,  1105,  1240,  1297,   119,   102,
             0,  

In [15]:
for i in range (10):
    print(tokenized_train_dataset[i])

{'labels': tensor(4), 'input_ids': tensor([  101,   173,  1197,   119,  2284,  2953,  3272,  1917,   178,  1440,
         1111,  1107,   170,  1704, 22351,   119,  1119,   112,   188,  3505,
         1105,  3123,  1106,  2037,  1106,  1443,  1217, 10063,  4404,   132,
         1119,   112,   188,  1579,  1113,  1159,  1107,  3195,  1117,  4420,
          132,  1119,   112,   188,  6559,  1114,   170,  1499,   118, 23555,
         2704,   113,   183,  9379,   114,  1134,  1139,  2153,  1138,  3716,
         1106,  1143,  1110,  1304,  1696,  1107,  1692,  1380,  5940,  1105,
         1128,  1444,  6059,   132,  1105,  1128,  1169,  1243,  5991, 16179,
         1106,  1267, 18137,  1443,  1515,  1106,  1267,  1140,  1148,   119,
         1541,   117,  1184,  1167,  1202,  1128,  1444,   136,   178,   112,
          182,  2807,  1303,  1774,  1106,  1341,  1104,  1251, 11344,   178,
         1138,  1164,  1140,   117,  1133,   178,   112,   182,  1541,  4619,
          170,  9153,   119, 


{'labels': tensor(1), 'input_ids': tensor([  101,  7595,   117,  1103,  9074,  1104,  1217,  1987,   119, 20029,
          112,   188,  5351,  1110,   170,  9488,  1104,  1103,  2541,   146,
          112,  1396,  1125,  1114,  1177,  1242,  1168,  8114,  1107, 17520,
          118,   118,  1363,  3995,   117,  6434,  2546,   119,  1135,  3093,
         1115,  1117,  2546,  2566,  1309,  6615,  1103,  2179,   119,  1135,
         1932,  2274,   123,  2005,  1104,  4892,  3516,  1106,  1243,  1126,
         2590,   119,  2627,  1144,  1159,  1111,  1115,  1137,  3349,  1106,
         2239,  1114,  1122,   136,   146,  1138,  1576,  1154,  1142,  2463,
         1114,  1242,  1168,  8114,  1105,   146,  1198,  1274,   112,   189,
         1243,  1122,   119,  1192,  1138,  1701,  3239,   117,  1128,  1138,
         4420,  1114,  2657,  2993,   117,  1725,  2762,   112,   189,  2256,
        10937,  1103,  2179,   136,  1135,   112,   188,  1107,  8178,  1643,
         1874, 10436, 24355,

In [16]:
tokenized_train_dataset[0].keys()

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])

In [25]:
num_labels = len(set([new_train_dataset[i]["label"] for i in range(len(new_train_dataset))]))
num_labels

5

### **Define Dataloaders**

In [17]:
train_dataloader = DataLoader(tokenized_train_dataset,shuffle=True, batch_size=2)
test_dataloader = DataLoader(tokenized_test_dataset, shuffle=False, batch_size=2)

### **Train the Model**

In [26]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased",num_labels = num_labels)

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torchvision\image.pyd'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING